In [3]:
###############
from md_Helpers.run_logs import configure_run_logging

configure_run_logging(
    True,
    notebook_name="Otto3_variable_Ncells_NPH",
)
###############

PosixPath('/exp/e961/data/MDsims-data/pnichols/run_logs/Otto3_variable_Ncells_NPH_2026-08-26_16-12-26.log')

In [ ]:
import re

from md_Helpers import hot_spike


densities = [0.720]
temperatures = [0.800]
energies = [600, 700, 800, 900, 1000]

radii = [3.0]
n_cells = [10, 20, 30, 40, 50, 60, 70, 80]
location_seeds = [6,7,8,9,10]


# Two-segment evolution settings
dt1 = 0.0005
nsteps1 = 200_000


# User choice
dt2 = 0.005
nsteps2 = 80_000


# NPH settings
pressure = None  # Infer pressure from the thermalized source
pressure_tail_samples = 100
tauS = 5.0
pressure_couple = "xyz"
barostat_gamma = 0.0

outer_mask_diameter_fraction = 0.75
nph_mask_controls_box = False

nph_volume_bounds = (0.75, 3.00)
nph_safety_check_period = 100


for n_cell in n_cells:
    for target_rho in densities:
        for kT in temperatures:
            for radius in radii:
                for injected_energy in energies:
                    for location_seed in location_seeds:
                        print(
                            f"\nNPH: cells={n_cell}, rho={target_rho}, "
                            f"kT={kT}, radius={radius}, "
                            f"energy={injected_energy}, "
                            f"location_seed={location_seed}"
                        )

                        try:
                            result = hot_spike.get_or_create_hot_spike(
                                n_fcc_cells=n_cell,
                                target_rho=target_rho,
                                kT=kT,
                                source_nsteps=1_000_000,
                                source_seed=1,

                                radius=radius,
                                injected_energy=injected_energy,
                                method="velocity_rescale_com",

                                dt1=dt1,
                                nsteps1=nsteps1,
                                dt2=dt2,
                                nsteps2=nsteps2,

                                log_period=1_000,
                                trajectory_period=1_000,

                                random_location=True,
                                location_seed=location_seed,

                                overwrite=False,
                                overwrite_initial=False,
                                overwrite_source=False,
                                create_source_if_missing=True,
                                reject_phase_separated_source=True,

                                ensemble="NPH",
                                pressure=pressure,
                                pressure_tail_samples=(
                                    pressure_tail_samples
                                ),
                                tauS=tauS,
                                pressure_couple=pressure_couple,
                                barostat_gamma=barostat_gamma,

                                outer_mask_diameter_fraction=(
                                    outer_mask_diameter_fraction
                                ),
                                nph_mask_controls_box=(
                                    nph_mask_controls_box
                                ),

                                nph_box_volume_ratio_bounds=(
                                    nph_volume_bounds
                                ),
                                nph_safety_check_period=(
                                    nph_safety_check_period
                                ),
                            )

                        except RuntimeError as error:
                            message = str(error)

                            if "NPH safety stop" not in message:
                                raise

                            match = re.search(
                                r"box volume ratio=([^ ]+)",
                                message,
                            )
                            if match is None:
                                raise

                            volume_ratio = float(match.group(1))

                            if volume_ratio > nph_volume_bounds[1]:
                                print(
                                    "Upper NPH volume limit reached. "
                                    "Assuming a bubble and continuing "
                                    "the sweep."
                                )
                                continue

                            # Do not classify a lower-volume stop as a bubble.
                            raise


NPH: cells=10, rho=0.72, kT=0.8, radius=3.0, energy=600, location_seed=6
Thermalized state exists: checking phase separation.
Loaded existing thermalized state:
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.720/kT_0.800/nsteps_1000000/seed_1/randomization.gsd
NPH pressure from thermalized source tail: -0.23006512 +/- 0.0489 (100 samples)
Thermalized state exists: checking phase separation.
Loaded existing thermalized state:
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.720/kT_0.800/nsteps_1000000/seed_1/randomization.gsd
Existing random-center excitation predates NPH recentering; rebuilding its saved initial frame.
Created new hot-spike initial state
state_path: /exp/e961/data/MDsims-data/pnichols/Excitation_States_v3/FCC/n_cells_10/source_kT_0.800/rho_0.720/source_nsteps_1000000/source_seed_1/method_velocity_rescale_com/radius_3.000/energy_600.000/random_center_seed_6/excitation_initial.gsd
creation_metadata_path: /exp